# Crack Width Viewer Demonstration

This notebook demonstrates the `CrackWidthViewer` which provides two interactive Plotly visualisations:

1. **3D Stem Plot** (`plot_load_cases`) — Crack widths as vertical stems rising from the M-N plane, with a translucent limit plane.
2. **2D Contour Map** (`plot_contours`) — Crack width field with a root-found `w_k = w_k,lim` boundary curve.

Both plots are also accessible directly from `CrackingCheck` via convenience wrappers.

### Design rationale

The plots intentionally do **not** show an M-N capacity envelope. ULS capacity and SLS crack widths
are fundamentally different limit states with different load factors — overlaying them would be
misleading. Instead, the `w_k = w_k,lim` boundary curve is the natural pass/fail boundary: load
cases on the safe side pass, those beyond it fail.

The domain bounds (M and N) come from the **ULS M-N interaction diagram** — SLS loads that exceed
ULS capacity are meaningless in a valid design. The boundary curve is computed via 1D root-finding
(`scipy.optimize.brentq`) on fixed-N slices, giving a crisp line independent of grid resolution.

By default, `force_cracked=True` — once a section has cracked under any load case, it remains
cracked under subsequent cases. This gives the most conservative (and realistic) crack width field.

In [ ]:
import warnings

# Section geometry
from materials.reinforced_concrete.geometry import (
    create_rectangular_section,
    create_linear_rebar_layer,
)

# Materials
from materials.reinforced_concrete.materials import ConcreteMaterial, Rebar

# Cracking check
from materials.reinforced_concrete.code_checks.ec2_2004 import (
    CrackingCheck,
    LoadDuration,
)

# Viewer (can also be used standalone)
from materials.reinforced_concrete.analysis.crack_width_viewer import CrackWidthViewer

## 1. Set Up a Section and Cracking Check

A 300 x 500 mm beam with 4H25 tension bars, 4H16 compression bars, and C30/37 concrete.

In [2]:
# Dimensions
width = 300   # mm
height = 500  # mm
cover = 35    # mm (to links)
link_dia = 10

section = create_rectangular_section(width=width, height=height)

# Tension reinforcement: 3H25 at bottom
tension_rebar = Rebar(diameter=25)
y_tens = cover + link_dia + tension_rebar.diameter / 2
side_cover = cover + link_dia
section.add_rebar_group(
    create_linear_rebar_layer(
        rebar=tension_rebar,
        n_bars=4,
        start_point=(side_cover + tension_rebar.diameter / 2, y_tens),
        end_point=(width - side_cover - tension_rebar.diameter / 2, y_tens),
    )
)

# Compression reinforcement: 2H16 at top
comp_rebar = Rebar(diameter=16)
y_comp = height - cover - link_dia - comp_rebar.diameter / 2
section.add_rebar_group(
    create_linear_rebar_layer(
        rebar=comp_rebar,
        n_bars=4,
        start_point=(side_cover + comp_rebar.diameter / 2, y_comp),
        end_point=(width - side_cover - comp_rebar.diameter / 2, y_comp),
    )
)

concrete = ConcreteMaterial(grade="C30/37")

# Cracking check (quasi-permanent, long-term)
check = CrackingCheck(
    section=section,
    concrete=concrete,
    w_k_limit=0.3,
    load_duration=LoadDuration.LONG_TERM,
    creep_coefficient=1.5,
)

M_cr = check.find_cracking_moment()
print(f"Section: {width}x{height} mm")
print(f"Concrete: {concrete.grade}")
print(f"Tension steel: 3H25 (A_s = {3 * tension_rebar.area:.0f} mm\u00b2)")
print(f"Cracking moment M_cr = {M_cr:.1f} kN\u00b7m")
print(f"Crack width limit w_k,lim = {check.w_k_limit} mm")

Section: 300x500 mm
Concrete: C30/37
Tension steel: 3H25 (A_s = 1473 mm²)
Cracking moment M_cr = 96.7 kN·m
Crack width limit w_k,lim = 0.3 mm


## 2. Define Load Cases

A mix of SLS load cases: some below the cracking moment, some above, and some that exceed the crack width limit.

In [3]:
load_cases = [
    {"name": "LC1: Low moment",           "M_Ed": 150, "N_Ed": 0},
    {"name": "LC2: Medium moment",        "M_Ed": 190, "N_Ed": 0},
    {"name": "LC3: High moment",          "M_Ed": 290, "N_Ed": 0},
    {"name": "LC4: Near limit",           "M_Ed": 230, "N_Ed": 0},
    {"name": "LC5: With compression",     "M_Ed": 230, "N_Ed": 500},
    {"name": "LC6: With tension",         "M_Ed": 150, "N_Ed": -500},
    {"name": "LC7: High M + compression", "M_Ed": 500, "N_Ed": 2000},
]

# Preview the crack widths
print(f"{'Load case':<30} {'M_Ed':>6} {'N_Ed':>6} {'w_k':>8} {'Status':>8}")
print("-" * 65)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for lc in load_cases:
        r = check.calculate_detailed(M_Ed=lc["M_Ed"], N_Ed=lc.get("N_Ed", 0))
        status = "PASS" if r.w_k <= r.w_k_limit else "FAIL"
        cracked = "cracked" if r.is_cracked else "uncracked"
        print(
            f"{lc['name']:<30} {lc['M_Ed']:>6.0f} {lc.get('N_Ed', 0):>6.0f} "
            f"{r.w_k:>8.3f} {status:>8}  ({cracked})"
        )

Load case                        M_Ed   N_Ed      w_k   Status
-----------------------------------------------------------------
LC1: Low moment                   150      0    0.181     PASS  (cracked)
LC2: Medium moment                190      0    0.240     PASS  (cracked)
LC3: High moment                  290      0    0.386     FAIL  (cracked)
LC4: Near limit                   230      0    0.299     PASS  (cracked)
LC5: With compression             230    500    0.176     PASS  (cracked)
LC6: With tension                 150   -500    0.336     FAIL  (cracked)
LC7: High M + compression         500   2000    0.286     PASS  (cracked)


## 3. Plot 1 — 3D Stem Plot (`plot_load_cases`)

Each load case appears as a vertical stem rising from the M-N plane (z = 0) to z = w_k.
A translucent orange plane marks the crack width limit.

- **Green** stems: w_k ≤ w_k,lim (pass)
- **Red** stems: w_k > w_k,lim (fail)

In [4]:
# Option A: via the CrackingCheck convenience wrapper
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fig_3d = check.plot_load_cases(
        load_cases,
        title="Crack Width \u2014 3D Stem Plot (300x500 Beam, C30/37)",
        show=False,
    )
fig_3d.show()

## 4. Plot 2 — 2D Contour Map (`plot_contours`)

A filled contour of crack width across the M-N domain. The dashed red line is the
`w_k = w_k,lim` boundary curve — computed via root-finding for each N slice. Load cases
on the safe side (lower crack widths) are green, those beyond the boundary are red.

Both M and N axes span the ULS M-N diagram bounds. `force_cracked=True` (default)
means crack widths are computed even below the cracking moment.

In [5]:
# Option A: via the CrackingCheck convenience wrapper
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fig_contour = check.plot_crack_width_contours(
        load_cases=load_cases,
        n_grid=25,
        title="Crack Width \u2014 M-N Contour Map (300x500 Beam, C30/37)",
        show=False,
    )
fig_contour.show()

## 5. Using the Viewer Directly

You can also instantiate `CrackWidthViewer` directly for more control.

In [ ]:
viewer = CrackWidthViewer(check)

# Just the contour, no load cases
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fig = viewer.plot_contours(
        n_grid=30,
        title="Crack Width Field (no load cases)",
        show=False,
    )
fig.show()

## 6. Effect of Crack Width Limit

Compare how the contour map changes for different exposure classes (w_k,lim = 0.2 vs 0.3 vs 0.4 mm).

In [7]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

limits = [0.2, 0.3, 0.4]

for w_lim in limits:
    check_lim = CrackingCheck(
        section=section,
        concrete=concrete,
        w_k_limit=w_lim,
        load_duration=LoadDuration.LONG_TERM,
        creep_coefficient=1.5,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fig = check_lim.plot_crack_width_contours(
            load_cases=load_cases,
            n_grid=20,
            title=f"w_k,lim = {w_lim} mm",
            show=False,
            width=700,
            height=500,
        )
    fig.show()

## 7. Effect of Creep Coefficient

Higher creep coefficients reduce E_cm,eff, which increases steel stress and crack widths.

In [8]:
creep_values = [0.0, 1.5, 3.0]

test_lcs = [
    {"name": "Service", "M_Ed": 220, "N_Ed": 0},
    {"name": "High M", "M_Ed": 260, "N_Ed": 0},
    {"name": "M + N",  "M_Ed": 220, "N_Ed": 300},
]

print(f"{'Creep coeff':>12} | {'Load case':<12} | {'E_cm,eff':>10} | {'w_k':>8} | {'Status':>6}")
print("-" * 65)

for phi in creep_values:
    check_phi = CrackingCheck(
        section=section,
        concrete=concrete,
        w_k_limit=0.3,
        load_duration=LoadDuration.LONG_TERM,
        creep_coefficient=phi,
        apply_nonlinear_creep=False,
    )
    for lc in test_lcs:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            r = check_phi.calculate_detailed(M_Ed=lc["M_Ed"], N_Ed=lc["N_Ed"])
        status = "PASS" if r.w_k <= 0.3 else "FAIL"
        print(
            f"{phi:>12.1f} | {lc['name']:<12} | {check_phi.E_cm_eff:>10.0f} | "
            f"{r.w_k:>8.3f} | {status:>6}"
        )

 Creep coeff | Load case    |   E_cm,eff |      w_k | Status
-----------------------------------------------------------------
         0.0 | Service      |      32837 |    0.295 |   PASS
         0.0 | High M       |      32837 |    0.355 |   FAIL
         0.0 | M + N        |      32837 |    0.212 |   PASS
         1.5 | Service      |      13135 |    0.284 |   PASS
         1.5 | High M       |      13135 |    0.343 |   FAIL
         1.5 | M + N        |      13135 |    0.207 |   PASS
         3.0 | Service      |       8209 |    0.271 |   PASS
         3.0 | High M       |       8209 |    0.329 |   FAIL
         3.0 | M + N        |       8209 |    0.197 |   PASS


## 8. 3D Stem Plot — Single Load Case Focus

The stem plot is also useful for highlighting a single critical load case in context.

In [9]:
single_lc = [{"name": "Critical LC", "M_Ed": 260, "N_Ed": 0}]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fig = check.plot_load_cases(
        single_lc,
        title="Single Load Case on M-N Envelope",
        show=False,
    )
fig.show()

## Summary

The `CrackWidthViewer` provides two complementary views of crack width behaviour:

| Method | Plot type | Best for |
|--------|-----------|----------|
| `plot_load_cases()` | 3D stem plot | Visualising discrete load cases against the limit |
| `plot_contours()` | 2D contour map | Understanding the crack width field across the full M-N domain |

Both methods are accessible:
- Directly: `CrackWidthViewer(check).plot_load_cases(...)` / `.plot_contours(...)`
- Via `CrackingCheck` wrappers: `check.plot_load_cases(...)` / `check.plot_crack_width_contours(...)`

### Key parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `n_grid` | 20 | Grid resolution for contour fill |
| `n_boundary_points` | 40 | Number of N slices for boundary root-finding |
| `force_cracked` | `True` | Compute w_k even when M < M_cr (section may already be cracked) |
| `load_cases` | `None` | Overlay discrete load case markers |
| `concrete_model_type` | `PARABOLA_RECTANGLE` | ULS concrete model for domain bounds |